In [ ]:
# -*- coding: utf-8 -*-
"""
Minimal scraper for eBay search page (first page only).
It collects name, price_raw, and url for each listing on the first page.
Uses requests + BeautifulSoup and robust selectors.
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import quote_plus

# ---------- 1) Configure your search URL ----------
# You can keep the tutorial's URL as-is, or build a new one with a keyword.
# Here we use the URL you pasted (searching for "mouse"):
BASE_URL = "https://www.ebay.com/sch/i.html?_from=R40&_nkw=mouse&_sacat=0"
# (Optional) If you want to force desktop layout with more items per page, you can append &_ipg=240

# ---------- 2) Download HTML with browser-like headers ----------
def fetch_html(url: str, timeout: int = 25) -> str:
    """
    Fetch the search page HTML using desktop-like headers.
    Adding common headers helps avoid anti-bot pages.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.ebay.com/",
        "DNT": "1",
        "Upgrade-Insecure-Requests": "1",
    }
    r = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
    r.raise_for_status()
    return r.text

# ---------- 3) Parse the first page for name / price / url ----------
def parse_first_page(html: str):
    """
    Parse eBay search results (first page only).
    Returns a list of dicts with keys: name, price_raw, url.
    Uses multiple fallbacks to be resilient to template changes.
    """
    soup = BeautifulSoup(html, "lxml")
    rows = []

    # Try several card containers (eBay has multiple templates)
    cards = soup.select("li.s-item")
    if not cards:
        cards = soup.select("div.s-item__wrapper")
    if not cards:
        cards = soup.select("div.s-item")

    for card in cards:
        # URL: prefer the listing link with '/itm/' in href
        a = card.select_one("a.s-item__link")
        if a is None:
            a = card.select_one("a[href*='/itm/']")
        if not a or not a.get("href"):
            continue
        url = a.get("href").split("?")[0]  # strip tracking query

        # NAME: try the title element, fallback to the anchor's text
        title_tag = (
            card.select_one("h3.s-item__title")
            or card.select_one("span[role='heading']")
            or a
        )
        name = title_tag.get_text(" ", strip=True) if title_tag else ""
        if not name or name.lower().startswith("shop on ebay"):
            # Skip ad shells / placeholders that have no real title
            continue

        # PRICE: standard price span; if missing we leave it empty
        price_tag = card.select_one("span.s-item__price")
        price_raw = price_tag.get_text(" ", strip=True) if price_tag else ""

        rows.append({"name": name, "price_raw": price_raw, "url": url})

    return rows

# ---------- 4) Orchestrator ----------
def scrape_ebay_first_page(base_url: str) -> pd.DataFrame:
    """
    Orchestrates: download → parse → return DataFrame.
    If you ever get zero rows, save HTML to inspect what's returned.
    """
    html = fetch_html(base_url)
    items = parse_first_page(html)

    # Debug aid: uncomment to inspect the raw page if you get 0 rows
    # with open("debug_search.html", "w", encoding="utf-8") as f:
    #     f.write(html)
    # print("HTML length:", len(html))

    return pd.DataFrame(items, columns=["name", "price_raw", "url"])

if __name__ == "__main__":
    df = scrape_ebay_first_page(BASE_URL)
    print(df.head(10))
    df.to_csv("ebay_first_page_mouse.csv", index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to ebay_first_page_mouse.csv")
